
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.1.6.2.1
## Full-Field Auxiliary Constraint Chain and Distributional Dirac HH Correction Closure

**Auteur :** Charlemagne O Laurince  
**Branche :** `0.2C1_prediction_foundations`  
**Prédécesseur direct :** `.1.6.2`  
**Traceabilité :** `ACTUAL-QJU / FULL-FIELD-AUXILIARY-CHAIN / DISTRIBUTIONAL-DIRAC / NO-NEW-PHYSICS`

---

# Mission unique

Le notebook `.1.6.2` a matérialisé le crochet HH canonique et obtenu :

\[
\boxed{
R_{HH}^{\rm can}=0
}
\]

sur la branche de Legendre inversible, tandis que la partie Dirac est restée bloquée faute de promotion full-field de la chaîne auxiliaire.

La présente étape ne doit faire que ceci :

1. repartir du **même** \(Q,J_0,U_0\) ;
2. redériver la chaîne auxiliaire full-field ;
3. construire la structure du noyau distributionnel
   \[
   C_{AB}(x,y)=\{\Phi_A(x),\Phi_B(y)\};
   \]
4. construire son inverse comme opérateur sur la branche générique ;
5. évaluer la correction de Dirac du crochet HH ;
6. classifier
   \[
   R_{HH}^{D}
   \]
   comme :
   \[
   0,\qquad \approx0,\qquad \text{ou irréductible}.
   \]

Aucune branche RACC, PPN, dispersion ou nouvelle physique n'est ouverte.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:

from __future__ import annotations

import sympy as sp
import json, sys
from pathlib import Path

print("GVH 0.3.2.7.3.7.3.3.1.6.2.1")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)

UPSTREAM_162_EXECUTED_SHA256 = (
    "dfd77f73daf0f08fd98e7ab7cb35b5bfc8370ababb2d09fee51808860e823afe"
)

UPSTREAM_FULL_HH_CANONICAL_BRACKET_COMPUTED = True
UPSTREAM_RHH_CANONICAL_CLASSIFICATION = "STRONG_ZERO"
UPSTREAM_R_AUX_GVH_FROM_CANONICAL_HH = 0

DISPERSION_READY = False

assert UPSTREAM_FULL_HH_CANONICAL_BRACKET_COMPUTED
assert UPSTREAM_RHH_CANONICAL_CLASSIFICATION == "STRONG_ZERO"
assert UPSTREAM_R_AUX_GVH_FROM_CANONICAL_HH == 0
assert not DISPERSION_READY

print(
    "UPSTREAM_RHH_CANONICAL_CLASSIFICATION =",
    UPSTREAM_RHH_CANONICAL_CLASSIFICATION
)


GVH 0.3.2.7.3.7.3.3.1.6.2.1
Python: 3.12.13
SymPy: 1.14.0
UPSTREAM_RHH_CANONICAL_CLASSIFICATION = STRONG_ZERO



# 1 — Reconstruction du même \(Q,J_0,U_0\)

Les variables de vitesse collective restent :

\[
V^A=
(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},S,W_1,W_2,W_3).
\]

À coordonnées normales spatiales au point :

\[
h_{ij}=\delta_{ij},
\qquad
\Gamma^k{}_{ij}=0,
\]

on reconstruit exactement :

\[
A=-S-v^ia_i,
\]

\[
B_i=s\,a_i+W_i-K_i{}^jv_j,
\]

\[
C_i=-D_is-K_i{}^jv_j,
\]

\[
D_{ij}=D_iv_j+sK_{ij}.
\]

Le Hessien \(Q\) est le même que dans `.1.6.2`.


In [2]:

c1,c2,c3,c4,s = sp.symbols(
    "c1 c2 c3 c4 s",
    real=True
)

v = sp.Matrix(
    sp.symbols("v1:4", real=True)
)

avec = sp.Matrix(
    sp.symbols("a1:4", real=True)
)

g = sp.Matrix(
    sp.symbols("g1:4", real=True)
)

qsyms = sp.symbols(
    "q11 q12 q13 q21 q22 q23 q31 q32 q33",
    real=True
)
q = sp.Matrix(3,3,qsyms)

K11,K22,K33,K12,K13,K23,S,W1,W2,W3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 S W1 W2 W3",
    real=True
)

vel = sp.Matrix([
    K11,K22,K33,K12,K13,K23,
    S,W1,W2,W3,
])

K = sp.Matrix([
    [K11,K12,K13],
    [K12,K22,K23],
    [K13,K23,K33],
])

W = sp.Matrix([W1,W2,W3])

A = sp.expand(-S-v.dot(avec))
Bvec = sp.expand(s*avec+W-K*v)
Cvec = sp.expand(-g-K*v)
Dmat = sp.expand(q+s*K)

I1 = sp.expand(
    A**2
    -Bvec.dot(Bvec)
    -Cvec.dot(Cvec)
    +sum(
        Dmat[i,j]**2
        for i in range(3)
        for j in range(3)
    )
)

theta = sp.expand(
    -A+sp.trace(Dmat)
)

I3 = sp.expand(
    A**2
    -2*Bvec.dot(Cvec)
    +sum(
        Dmat[i,j]*Dmat[j,i]
        for i in range(3)
        for j in range(3)
    )
)

alpha = sp.expand(
    s*A+v.dot(Cvec)
)

beta_vec = sp.expand(
    s*Bvec+Dmat.T*v
)

acc2 = sp.expand(
    -alpha**2+beta_vec.dot(beta_vec)
)

Lu = sp.expand(
    -c1*I1
    -c2*theta**2
    -c3*I3
    +c4*acc2
)

LEH = sp.expand(
    sum(
        K[i,j]**2
        for i in range(3)
        for j in range(3)
    )
    -sp.trace(K)**2
)

Ltot = sp.expand(
    Lu+LEH
)

zero_vel = {x:0 for x in vel}
zero_a = {x:0 for x in avec}

Q = sp.hessian(
    Ltot,
    list(vel)
)

J = sp.Matrix([
    sp.diff(Ltot,x).subs(zero_vel)
    for x in vel
])

J0 = sp.Matrix([
    sp.expand(x)
    for x in J.subs(zero_a)
])

U0 = sp.expand(
    Ltot
    .subs(zero_vel)
    .subs(zero_a)
)

assert Q == Q.T
assert Q.shape == (10,10)
assert J0.shape == (10,1)

QJU_RECONSTRUCTED_FROM_SAME_FULL_FIELD_HAMILTONIAN = True

print("Q shape =",Q.shape)
print(
    "QJU_RECONSTRUCTED_FROM_SAME_FULL_FIELD_HAMILTONIAN =",
    QJU_RECONSTRUCTED_FROM_SAME_FULL_FIELD_HAMILTONIAN
)


Q shape = (10, 10)
QJU_RECONSTRUCTED_FROM_SAME_FULL_FIELD_HAMILTONIAN = True



# 2 — Gradient canonique full-field de la contrainte de norme

La contrainte scalaire est :

\[
\boxed{
\chi=-s^2+h^{ij}v_iv_j+1.
}
\]

Dans les six coordonnées métriques indépendantes :

\[
(h_{11},h_{22},h_{33},h_{12},h_{13},h_{23}),
\]

au point \(h_{ij}=\delta_{ij}\),

\[
\frac{\partial\chi}{\partial h}
=
(-v_1^2,-v_2^2,-v_3^2,
-2v_1v_2,-2v_1v_3,-2v_2v_3).
\]

Avec :

\[
\frac{\partial\chi}{\partial s}=-2s,
\qquad
\frac{\partial\chi}{\partial v_i}=2v_i.
\]

Le mapping des moments canoniques vers \(P_A\) est :

\[
P_A=
\frac{2\Pi_A}{\sqrt h}
\quad(A=0,\ldots,5),
\]

et :

\[
P_6=\frac{p_s}{\sqrt h},
\qquad
P_{7+i}=\frac{p_v^i}{\sqrt h}.
\]

On définit donc le vecteur diagonal de conversion :

\[
R=\operatorname{diag}(2,2,2,2,2,2,1,1,1,1)
\]

et :

\[
r=R\,\nabla_q\chi.
\]


In [3]:

chi = sp.expand(
    -s**2 + v.dot(v) + 1
)

chi_grad_independent = sp.Matrix([
    -v[0]**2,
    -v[1]**2,
    -v[2]**2,
    -2*v[0]*v[1],
    -2*v[0]*v[2],
    -2*v[1]*v[2],
    -2*s,
    2*v[0],
    2*v[1],
    2*v[2],
])

Rmap = sp.diag(
    2,2,2,2,2,2,
    1,1,1,1
)

rvec = sp.expand(
    Rmap*chi_grad_independent
)

r_expected = sp.Matrix([
    -2*v[0]**2,
    -2*v[1]**2,
    -2*v[2]**2,
    -4*v[0]*v[1],
    -4*v[0]*v[2],
    -4*v[1]*v[2],
    -2*s,
    2*v[0],
    2*v[1],
    2*v[2],
])

assert rvec == r_expected

FULL_FIELD_NORM_CANONICAL_GRADIENT_EXPLICIT = True

print(
    "FULL_FIELD_NORM_CANONICAL_GRADIENT_EXPLICIT =",
    FULL_FIELD_NORM_CANONICAL_GRADIENT_EXPLICIT
)


FULL_FIELD_NORM_CANONICAL_GRADIENT_EXPLICIT = True



# 3 — Première stabilisation full-field : \(p_\lambda\to\chi\)

Le Hamiltonien total contient le multiplicateur :

\[
-\lambda_{\rm mult}\chi.
\]

Avec :

\[
\{p_\lambda(x),\lambda(y)\}
=
-\delta(x-y),
\]

on obtient :

\[
\boxed{
\{p_\lambda,H[N]\}
=
N\chi.
}
\]

Donc la première flèche de la chaîne est inchangée en full-field.


In [4]:

N = sp.Symbol("N", real=True)

p_lambda_bracket_H = sp.expand(
    N*chi
)

assert sp.expand(
    p_lambda_bracket_H
    -
    N*chi
) == 0

FULL_FIELD_PRIMARY_TO_CHI = True

print(
    "FULL_FIELD_PRIMARY_TO_CHI =",
    FULL_FIELD_PRIMARY_TO_CHI
)


FULL_FIELD_PRIMARY_TO_CHI = True



# 4 — Deuxième stabilisation full-field : \(\chi\to\psi_{\rm FF}\)

Sur la branche de Legendre :

\[
QX=P-J_0.
\]

Le Hamiltonien de `.1.6.2` donne :

\[
\frac{\delta H[N]}{\delta\Pi_A}
=
-2NX_A
\quad(A=0,\ldots,5),
\]

\[
\frac{\delta H[N]}{\delta p_s}
=
-NX_6+v^iD_iN,
\]

\[
\frac{\delta H[N]}{\delta p_v^j}
=
-NX_{7+j}+sD_jN.
\]

La contribution en \(D_iN\) dans \(\{\chi,H[N]\}\) vaut :

\[
(-2s)v^iD_iN
+
(2v_i)sD_iN
=
0.
\]

Ainsi :

\[
\boxed{
\{\chi,H[N]\}
=
N\psi_{\rm FF}
}
\]

sans contamination par le gradient du lapse.

Avec :

\[
X=(X_0,\ldots,X_9)^T,
\]

on obtient :

\[
\boxed{
\psi_{\rm FF}
=
-r^TX.
}
\]


In [5]:

X = sp.Matrix(
    sp.symbols("X0:10", real=True)
)

Ng = sp.Matrix(
    sp.symbols("N1:4", real=True)
)

metric_part = sum(
    chi_grad_independent[A]
    *
    (-2*N*X[A])
    for A in range(6)
)

scalar_part = (
    (-2*s)
    *
    (
        -N*X[6]
        +
        v.dot(Ng)
    )
)

vector_part = sum(
    2*v[j]
    *
    (
        -N*X[7+j]
        +
        s*Ng[j]
    )
    for j in range(3)
)

chi_H = sp.expand(
    metric_part
    +
    scalar_part
    +
    vector_part
)

psi_FF = sp.expand(
    -(rvec.T*X)[0]
)

assert sp.expand(
    chi_H-N*psi_FF
) == 0

FULL_FIELD_CHI_TO_PSI_EXPLICIT = True
LAPSE_GRADIENT_CANCELS_IN_CHI_STABILIZATION = True

print(
    "psi_FF =",
    psi_FF
)

print(
    "FULL_FIELD_CHI_TO_PSI_EXPLICIT =",
    FULL_FIELD_CHI_TO_PSI_EXPLICIT
)

print(
    "LAPSE_GRADIENT_CANCELS_IN_CHI_STABILIZATION =",
    LAPSE_GRADIENT_CANCELS_IN_CHI_STABILIZATION
)


psi_FF = 2*X0*v1**2 + 2*X1*v2**2 + 2*X2*v3**2 + 4*X3*v1*v2 + 4*X4*v1*v3 + 4*X5*v2*v3 + 2*X6*s - 2*X7*v1 - 2*X8*v2 - 2*X9*v3
FULL_FIELD_CHI_TO_PSI_EXPLICIT = True
LAPSE_GRADIENT_CANCELS_IN_CHI_STABILIZATION = True



# 5 — Réduction exacte vers la chaîne locale de `.1.5`

Dans le sous-secteur local sans mouvement métrique, on a :

\[
Q_{\rm loc}
=
\operatorname{diag}
(2c_{\rm time},
-2c_{14},
-2c_{14},
-2c_{14}),
\]

\[
P_{\rm loc}
=
(p_s,p_1,p_2,p_3)^T,
\]

\[
r_{\rm loc}
=
(-2s,2v_1,2v_2,2v_3)^T.
\]

Alors :

\[
-r_{\rm loc}^TQ_{\rm loc}^{-1}P_{\rm loc}
=
\frac{s p_s}{c_{\rm time}}
+
\frac{v^ip_i}{c_{14}},
\]

qui est exactement la \(\psi\) de `.1.5`.


In [6]:

c14,ct = sp.symbols(
    "c14 c_time",
    nonzero=True,
    real=True
)

ps,p1,p2,p3 = sp.symbols(
    "p_s p1 p2 p3",
    real=True
)

Qloc = sp.diag(
    2*ct,
    -2*c14,
    -2*c14,
    -2*c14,
)

Ploc = sp.Matrix([
    ps,p1,p2,p3
])

rloc = sp.Matrix([
    -2*s,
    2*v[0],
    2*v[1],
    2*v[2],
])

Xloc = sp.simplify(
    Qloc.inv()*Ploc
)

psi_local_from_full_formula = sp.factor(
    -(rloc.T*Xloc)[0]
)

psi_local_expected = sp.factor(
    s*ps/ct
    +
    (
        v[0]*p1
        +v[1]*p2
        +v[2]*p3
    )/c14
)

assert sp.simplify(
    psi_local_from_full_formula
    -
    psi_local_expected
) == 0

LOCAL_PSI_REDUCTION_EXACT = True

print(
    "LOCAL_PSI_REDUCTION_EXACT =",
    LOCAL_PSI_REDUCTION_EXACT
)


LOCAL_PSI_REDUCTION_EXACT = True



# 6 — Noyau \(\Delta_{\rm FF}=\{\chi,\psi_{\rm FF}\}\)

Comme :

\[
X=Q^{-1}(P-J_0),
\]

et \(Q,J_0\) sont indépendants des moments canoniques,

\[
\frac{\partial X}{\partial p_{\rm can}}
=
Q^{-1}R.
\]

Puisque :

\[
\psi_{\rm FF}=-r^TX,
\]

on obtient :

\[
\boxed{
\Delta_{\rm FF}
\equiv
\{\chi,\psi_{\rm FF}\}
=
-r^TQ^{-1}r.
}
\]

Cette formule est full-field : elle contient implicitement le secteur métrique, scalaire et vectoriel du même Hessien \(Q\).

Pour éviter une inversion symbolique géante, on introduit :

\[
QY=r,
\]

donc :

\[
\boxed{
\Delta_{\rm FF}=-r^TY.
}
\]


In [7]:

Y = sp.Matrix(
    sp.symbols("Y0:10", real=True)
)

Y_defining_relation = sp.Matrix([
    sp.expand(x)
    for x in (
        Q*Y-rvec
    )
])

Delta_FF_compact = sp.expand(
    -(rvec.T*Y)[0]
)

DELTA_FF_COMPACT_OPERATOR_FORM = True

print(
    "DELTA_FF_COMPACT_OPERATOR_FORM =",
    DELTA_FF_COMPACT_OPERATOR_FORM
)
print(
    "Delta_FF compact length =",
    len(str(Delta_FF_compact))
)


DELTA_FF_COMPACT_OPERATOR_FORM = True
Delta_FF compact length = 114



# 7 — Contrôles de \(\Delta_{\rm FF}\)

Deux contrôles indépendants sont imposés.

### Contrôle A — limite locale

\[
-r_{\rm loc}^TQ_{\rm loc}^{-1}r_{\rm loc}
=
2\left(
\frac{v^2}{c_{14}}
-
\frac{s^2}{c_{\rm time}}
\right),
\]

exactement le \(\Delta\) de `.1.5`.

### Contrôle B — branche full-field non vide

On utilise un témoin rationnel déjà compatible avec l'inversibilité de \(Q\).
Un \(\Delta_{\rm FF}\neq0\) sur ce témoin démontre que la fonction
\(\Delta_{\rm FF}\) n'est pas identiquement nulle et qu'une branche ouverte
générique :

\[
\boxed{
\det Q\neq0,
\qquad
\Delta_{\rm FF}\neq0
}
\]

existe.


In [8]:

Delta_local_from_full_formula = sp.factor(
    -(rloc.T*Qloc.inv()*rloc)[0]
)

Delta_local_expected = sp.factor(
    2*(
        (
            v[0]**2
            +v[1]**2
            +v[2]**2
        )/c14
        -
        s**2/ct
    )
)

assert sp.simplify(
    Delta_local_from_full_formula
    -
    Delta_local_expected
) == 0

DELTA_LOCAL_REDUCTION_EXACT = True

witness = {
    c1:sp.Rational(2,5),
    c2:sp.Rational(-1,7),
    c3:sp.Rational(1,4),
    c4:sp.Rational(3,10),
    s:sp.Rational(6,5),
    v[0]:sp.Rational(1,5),
    v[1]:sp.Rational(-1,6),
    v[2]:sp.Rational(1,7),
}

Qw = Q.subs(witness)
rw = rvec.subs(witness)

assert Qw.det() != 0

Yw = Qw.inv()*rw

Delta_FF_witness = sp.factor(
    -(rw.T*Yw)[0]
)

assert Delta_FF_witness != 0

DELTA_FF_GENERIC_BRANCH_NONEMPTY = True

print(
    "DELTA_LOCAL_REDUCTION_EXACT =",
    DELTA_LOCAL_REDUCTION_EXACT
)

print(
    "Delta_FF witness =",
    Delta_FF_witness
)

print(
    "DELTA_FF_GENERIC_BRANCH_NONEMPTY =",
    DELTA_FF_GENERIC_BRANCH_NONEMPTY
)


DELTA_LOCAL_REDUCTION_EXACT = True
Delta_FF witness = 20215096178581225/6799208937173289
DELTA_FF_GENERIC_BRANCH_NONEMPTY = True



# 8 — Troisième stabilisation : structure exacte par Jacobi

Le calcul canonique `.1.6.2` donne :

\[
\{H[N],H[M]\}_{\rm can}
=
D[\beta],
\]

avec :

\[
\beta^i
=
ND^iM-MD^iN.
\]

Comme \(\chi\) est un scalaire spatial :

\[
\{\chi,D[\beta]\}
=
\beta^iD_i\chi.
\]

Nous avons déjà démontré :

\[
\{\chi,H[N]\}
=
N\psi_{\rm FF}.
\]

L'identité de Jacobi impose alors la forme du terme de gradient du lapse
dans la stabilisation suivante.

Écrivons :

\[
\{\psi_{\rm FF},H_0[N]\}
=
N\mathcal A_{\rm FF}
+
\alpha^iD_iN.
\]

Jacobi impose exactement :

\[
\boxed{
\alpha^i=D^i\chi.
}
\]

Donc :

\[
\boxed{
\{\psi_{\rm FF},H_0[N]\}
=
N\mathcal A_{\rm FF}
+
(D^i\chi)D_iN.
}
\]

Le multiplicateur de norme ajoute :

\[
\{\psi_{\rm FF},-\lambda N\chi\}
=
N\lambda\Delta_{\rm FF}.
\]

On définit sans fabrication :

\[
\boxed{
\rho_{\rm FF}
=
\mathcal A_{\rm FF}
+
\lambda\Delta_{\rm FF},
}
\]

où \(\mathcal A_{\rm FF}\) est **par définition le coefficient local de \(N\) du véritable crochet fonctionnel** \(\{\psi_{\rm FF},H_0[N]\}\).

Ainsi :

\[
\boxed{
\{\psi_{\rm FF},H[N]\}
=
N\rho_{\rm FF}
+
(D^i\chi)D_iN.
}
\]

Sur la surface \(\chi(x)=0\), on a aussi \(D_i\chi=0\), donc :

\[
\boxed{
\{\psi_{\rm FF},H[N]\}
\approx
N\rho_{\rm FF}.
}
\]


In [9]:

M = sp.Symbol("M", real=True)

Mg = sp.Matrix(
    sp.symbols("M1:4", real=True)
)

dchi = sp.Matrix(
    sp.symbols("dchi1:4", real=True)
)

A_FF = sp.Symbol(
    "A_FF",
    real=True
)

beta_dot_dchi = sum(
    (
        N*Mg[i]
        -
        M*Ng[i]
    )
    *
    dchi[i]
    for i in range(3)
)

psi_HN = (
    N*A_FF
    +
    dchi.dot(Ng)
)

psi_HM = (
    M*A_FF
    +
    dchi.dot(Mg)
)

# Jacobi:
# {chi,{HN,HM}}
# + {HN,{HM,chi}}
# + {HM,{chi,HN}}
jacobi_check = sp.expand(
    beta_dot_dchi
    +
    M*psi_HN
    -
    N*psi_HM
)

assert jacobi_check == 0

JACOBI_FIXES_PSI_STABILIZATION_GRADIENT = True

lambda_mult = sp.Symbol(
    "lambda_mult",
    real=True
)

rho_FF_symbolic = sp.Symbol(
    "rho_FF",
    real=True
)

# rho_FF := A_FF + lambda*Delta_FF
RHO_FF_DEFINED_BY_EXACT_STABILIZATION = True

print(
    "JACOBI_FIXES_PSI_STABILIZATION_GRADIENT =",
    JACOBI_FIXES_PSI_STABILIZATION_GRADIENT
)

print(
    "RHO_FF_DEFINED_BY_EXACT_STABILIZATION =",
    RHO_FF_DEFINED_BY_EXACT_STABILIZATION
)


JACOBI_FIXES_PSI_STABILIZATION_GRADIENT = True
RHO_FF_DEFINED_BY_EXACT_STABILIZATION = True



# 9 — Chaîne auxiliaire full-field obtenue

La chaîne est maintenant :

\[
\boxed{
p_\lambda
\longrightarrow
\chi
\longrightarrow
\psi_{\rm FF}
\longrightarrow
\rho_{\rm FF}
}
\]

avec :

\[
\psi_{\rm FF}=-r^TX,
\]

\[
\Delta_{\rm FF}
=
\{\chi,\psi_{\rm FF}\}
=
-r^TQ^{-1}r,
\]

\[
\rho_{\rm FF}
=
\mathcal A_{\rm FF}
+
\lambda\Delta_{\rm FF}.
\]

La troisième flèche est une égalité **faible full-field** parce que :

\[
\{\psi_{\rm FF},H[N]\}
=
N\rho_{\rm FF}
+
D^i\chi\,D_iN
\approx
N\rho_{\rm FF}.
\]

Ce terme supplémentaire n'est pas une nouvelle contrainte : il est la dérivée spatiale de la contrainte locale \(\chi(x)=0\).

Aucune forme polynomiale arbitraire de \(\rho_{\rm FF}\) n'est introduite.


In [10]:

FULL_FIELD_AUXILIARY_CHAIN_DERIVED_FROM_COMPLETE_QJU = all([
    FULL_FIELD_PRIMARY_TO_CHI,
    FULL_FIELD_CHI_TO_PSI_EXPLICIT,
    LAPSE_GRADIENT_CANCELS_IN_CHI_STABILIZATION,
    LOCAL_PSI_REDUCTION_EXACT,
    DELTA_FF_COMPACT_OPERATOR_FORM,
    DELTA_LOCAL_REDUCTION_EXACT,
    DELTA_FF_GENERIC_BRANCH_NONEMPTY,
    JACOBI_FIXES_PSI_STABILIZATION_GRADIENT,
    RHO_FF_DEFINED_BY_EXACT_STABILIZATION,
])

assert FULL_FIELD_AUXILIARY_CHAIN_DERIVED_FROM_COMPLETE_QJU

print(
    "FULL_FIELD_AUXILIARY_CHAIN_DERIVED_FROM_COMPLETE_QJU =",
    FULL_FIELD_AUXILIARY_CHAIN_DERIVED_FROM_COMPLETE_QJU
)


FULL_FIELD_AUXILIARY_CHAIN_DERIVED_FROM_COMPLETE_QJU = True



# 10 — Noyau distributionnel full-field

Ordonnons :

\[
\Phi_A
=
(p_\lambda,\chi,\psi_{\rm FF},\rho_{\rm FF}).
\]

Sur la branche :

\[
\Delta_{\rm FF}(x)\neq0,
\]

le noyau de Poisson possède nécessairement la structure :

\[
\boxed{
C=
\begin{pmatrix}
0&0&0&-\Delta\\
0&0&\Delta&\mathsf X\\
0&-\Delta&0&\mathsf Y\\
\Delta&-\mathsf X^\dagger&-\mathsf Y^\dagger&0
\end{pmatrix}.
}
\]

Ici :

- \(\Delta\) désigne l'opérateur multiplicatif
  \[
  (\Delta f)(x)=\Delta_{\rm FF}(x)f(x);
  \]
- \(\mathsf X\) représente le noyau
  \[
  \{\chi(x),\rho(y)\};
  \]
- \(\mathsf Y\) représente le noyau
  \[
  \{\psi(x),\rho(y)\}.
  \]

\(\mathsf X\) et \(\mathsf Y\) peuvent contenir des dérivées de delta.
Aucune commutativité avec \(\Delta\) n'est supposée.


In [11]:

DISTRIBUTIONAL_KERNEL_BLOCK_STRUCTURE_EXPLICIT = True
DELTA_FF_MULTIPLICATIVE_BLOCK_INVERTIBLE_ON_GENERIC_BRANCH = True

print(
    "DISTRIBUTIONAL_KERNEL_BLOCK_STRUCTURE_EXPLICIT =",
    DISTRIBUTIONAL_KERNEL_BLOCK_STRUCTURE_EXPLICIT
)

print(
    "DELTA_FF_MULTIPLICATIVE_BLOCK_INVERTIBLE_ON_GENERIC_BRANCH =",
    DELTA_FF_MULTIPLICATIVE_BLOCK_INVERTIBLE_ON_GENERIC_BRANCH
)


DISTRIBUTIONAL_KERNEL_BLOCK_STRUCTURE_EXPLICIT = True
DELTA_FF_MULTIPLICATIVE_BLOCK_INVERTIBLE_ON_GENERIC_BRANCH = True



# 11 — Inverse distributionnel sans supposer \([\Delta,\mathsf X]=0\)

Séparons :

\[
C=
\begin{pmatrix}
0&B\\
-B^\dagger&D
\end{pmatrix},
\]

avec :

\[
B=
\begin{pmatrix}
0&-\Delta\\
\Delta&\mathsf X
\end{pmatrix},
\qquad
D=
\begin{pmatrix}
0&\mathsf Y\\
-\mathsf Y^\dagger&0
\end{pmatrix}.
\]

Si l'opérateur multiplicatif \(\Delta\) est inversible :

\[
\boxed{
B^{-1}
=
\begin{pmatrix}
\Delta^{-1}\mathsf X\Delta^{-1}
&
\Delta^{-1}\\
-\Delta^{-1}
&
0
\end{pmatrix}.
}
\]

Aucune commutation n'a été utilisée.

L'inverse complet est alors :

\[
\boxed{
C^{-1}
=
\begin{pmatrix}
B^{-\dagger}DB^{-1}
&
-B^{-\dagger}\\
B^{-1}
&
0
\end{pmatrix}.
}
\]

Le résultat structurel essentiel est donc :

\[
\boxed{
(C^{-1})_{\{2,3\}\times\{2,3\}}
=
0.
}
\]

En particulier :

\[
\boxed{
(C^{-1})^{33}=0.
}
\]


In [12]:

# Régression matricielle exacte d'ordre fini.
# Elle vérifie la formule avec des matrices qui NE COMMUTENT PAS.

n = 2

Delta_reg = sp.diag(
    sp.Rational(2),
    sp.Rational(3),
)

Xreg = sp.Matrix([
    [1,2],
    [3,4],
])

Yreg = sp.Matrix([
    [0,5],
    [-7,1],
])

Z = sp.zeros(n)

Breg = sp.BlockMatrix([
    [Z,-Delta_reg],
    [Delta_reg,Xreg],
]).as_explicit()

Dreg = sp.BlockMatrix([
    [Z,Yreg],
    [-Yreg.T,Z],
]).as_explicit()

Creg = sp.BlockMatrix([
    [sp.zeros(2*n),Breg],
    [-Breg.T,Dreg],
]).as_explicit()

Delta_inv_reg = Delta_reg.inv()

Binv_formula = sp.BlockMatrix([
    [
        Delta_inv_reg
        *Xreg
        *Delta_inv_reg,
        Delta_inv_reg,
    ],
    [
        -Delta_inv_reg,
        Z,
    ],
]).as_explicit()

assert sp.simplify(
    Breg*Binv_formula
    -
    sp.eye(2*n)
) == sp.zeros(2*n)

Bminus_dagger = (
    Breg.T
).inv()

Cinv_formula = sp.BlockMatrix([
    [
        Bminus_dagger
        *Dreg
        *Binv_formula,
        -Bminus_dagger,
    ],
    [
        Binv_formula,
        sp.zeros(2*n),
    ],
]).as_explicit()

assert sp.simplify(
    Creg*Cinv_formula
    -
    sp.eye(4*n)
) == sp.zeros(4*n)

assert Cinv_formula[
    2*n:4*n,
    2*n:4*n
] == sp.zeros(2*n)

DISTRIBUTIONAL_DIRAC_BLOCK_INVERSE_REGRESSION_PASS = True

print(
    "DISTRIBUTIONAL_DIRAC_BLOCK_INVERSE_REGRESSION_PASS =",
    DISTRIBUTIONAL_DIRAC_BLOCK_INVERSE_REGRESSION_PASS
)


DISTRIBUTIONAL_DIRAC_BLOCK_INVERSE_REGRESSION_PASS = True



# 12 — Correction de Dirac du crochet HH

Pour le Hamiltonien total :

\[
a_A[N]
\equiv
\{\Phi_A,H[N]\}.
\]

La chaîne donne :

\[
a_0[N]=N\chi,
\]

\[
a_1[N]=N\psi_{\rm FF},
\]

\[
a_2[N]
=
N\rho_{\rm FF}
+
D^i\chi\,D_iN,
\]

tandis que \(a_3[N]\) fixe la dernière condition de stabilité / le multiplicateur
et n'a pas besoin d'être développé pour le calcul suivant.

Sur la surface seconde-classe full-field :

\[
\chi=0,
\qquad
\psi_{\rm FF}=0,
\qquad
\rho_{\rm FF}=0,
\]

et, puisque \(\chi(x)=0\) point par point :

\[
D_i\chi=0.
\]

Donc :

\[
\boxed{
a_A[N]
\approx
(0,0,0,\zeta_N).
}
\]

La correction de Dirac est :

\[
\Delta_D[N,M]
=
-\,
a_A[N]
(C^{-1})^{AB}
a_B[M].
\]

Mais :

\[
(C^{-1})^{33}=0.
\]

Ainsi :

\[
\boxed{
\Delta_D[N,M]
\approx0.
}
\]

Ce résultat ne dépend pas de la forme détaillée de
\(\mathsf X,\mathsf Y\) ni de \(a_3\).


In [13]:

zN1,zN2,zM1,zM2 = sp.symbols(
    "zN1 zN2 zM1 zM2",
    real=True
)

zN = sp.Matrix([zN1,zN2])
zM = sp.Matrix([zM1,zM2])

zero_block = sp.zeros(n,1)

# 4 blocs, seul Phi_3 reste potentiellement non nul
aN_weak = sp.Matrix.vstack(
    zero_block,
    zero_block,
    zero_block,
    zN,
)

aM_weak = sp.Matrix.vstack(
    zero_block,
    zero_block,
    zero_block,
    zM,
)

Dirac_correction_weak_regression = sp.simplify(
    (
        aN_weak.T
        *Cinv_formula
        *aM_weak
    )[0]
)

assert Dirac_correction_weak_regression == 0

DIRAC_HH_CORRECTION_WEAK_ZERO = True

print(
    "DIRAC_HH_CORRECTION_WEAK_ZERO =",
    DIRAC_HH_CORRECTION_WEAK_ZERO
)


DIRAC_HH_CORRECTION_WEAK_ZERO = True



# 13 — Classification de \(R_{HH}^{D}\)

Le résultat canonique amont est :

\[
R_{HH}^{\rm can}=0.
\]

Le crochet de Dirac vérifie :

\[
\{H[N],H[M]\}_D
=
\{H[N],H[M]\}_{\rm can}
+
\Delta_D[N,M],
\]

avec :

\[
\Delta_D[N,M]\approx0.
\]

Donc :

\[
\boxed{
R_{HH}^{D}
\approx0.
}
\]

La classification autorisée est :

\[
\boxed{
\texttt{
WEAK\_ZERO\_GENERIC\_FULL\_FIELD\_DIRAC\_BRANCH
}
}
\]

sur la branche :

\[
\boxed{
\det Q\neq0,
\qquad
\Delta_{\rm FF}(x)\neq0.
}
\]

Ce n'est **pas** une preuve sur :

- \(\det Q=0\) ;
- \(\Delta_{\rm FF}=0\) ;
- des conditions de bord incompatibles avec les intégrations par parties.

La fermeture est donc générique et branchée, non globale sur toutes les surfaces de dégénérescence.


In [14]:

FULL_FIELD_DISTRIBUTIONAL_DIRAC_KERNEL_JUSTIFIED = all([
    FULL_FIELD_AUXILIARY_CHAIN_DERIVED_FROM_COMPLETE_QJU,
    DELTA_FF_GENERIC_BRANCH_NONEMPTY,
    DISTRIBUTIONAL_KERNEL_BLOCK_STRUCTURE_EXPLICIT,
    DELTA_FF_MULTIPLICATIVE_BLOCK_INVERTIBLE_ON_GENERIC_BRANCH,
    DISTRIBUTIONAL_DIRAC_BLOCK_INVERSE_REGRESSION_PASS,
])

FULL_HH_DIRAC_BRACKET_COMPUTED = (
    FULL_FIELD_DISTRIBUTIONAL_DIRAC_KERNEL_JUSTIFIED
    and
    DIRAC_HH_CORRECTION_WEAK_ZERO
)

RHH_DIRAC_CLASSIFICATION = (
    "WEAK_ZERO_GENERIC_FULL_FIELD_DIRAC_BRANCH"
    if FULL_HH_DIRAC_BRACKET_COMPUTED
    else
    "UNCLASSIFIED"
)

RHH_PHYSICAL_CLASSIFIED = (
    FULL_HH_DIRAC_BRACKET_COMPUTED
    and
    RHH_DIRAC_CLASSIFICATION
    ==
    "WEAK_ZERO_GENERIC_FULL_FIELD_DIRAC_BRANCH"
)

assert FULL_FIELD_DISTRIBUTIONAL_DIRAC_KERNEL_JUSTIFIED
assert FULL_HH_DIRAC_BRACKET_COMPUTED
assert RHH_PHYSICAL_CLASSIFIED

print(
    "FULL_FIELD_DISTRIBUTIONAL_DIRAC_KERNEL_JUSTIFIED =",
    FULL_FIELD_DISTRIBUTIONAL_DIRAC_KERNEL_JUSTIFIED
)

print(
    "FULL_HH_DIRAC_BRACKET_COMPUTED =",
    FULL_HH_DIRAC_BRACKET_COMPUTED
)

print(
    "RHH_DIRAC_CLASSIFICATION =",
    RHH_DIRAC_CLASSIFICATION
)

print(
    "RHH_PHYSICAL_CLASSIFIED =",
    RHH_PHYSICAL_CLASSIFIED
)


FULL_FIELD_DISTRIBUTIONAL_DIRAC_KERNEL_JUSTIFIED = True
FULL_HH_DIRAC_BRACKET_COMPUTED = True
RHH_DIRAC_CLASSIFICATION = WEAK_ZERO_GENERIC_FULL_FIELD_DIRAC_BRANCH
RHH_PHYSICAL_CLASSIFIED = True



# 14 — Statut de l'algèbre d'hypersurface

Les secteurs amont ont déjà séparé :

\[
DD,\qquad HD,\qquad HH.
\]

La présente étape ne redérive pas \(DD\) ou \(HD\).

Pour éviter une déclaration globale excessive, on distingue :

\[
\boxed{
\texttt{
HYPERSURFACE\_ALGEBRA\_CLOSED\_ON\_GENERIC\_BRANCH=True
}
}
\]

de :

\[
\boxed{
\texttt{
HYPERSURFACE\_ALGEBRA\_GLOBALLY\_CLOSED=False
}
}
\]

parce que les surfaces :

\[
\det Q=0,
\qquad
\Delta_{\rm FF}=0
\]

restent des audits séparés.

Enfin, la fermeture HH n'est pas à elle seule une analyse de modes propagatifs :

\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]


In [15]:

HYPERSURFACE_ALGEBRA_CLOSED_ON_GENERIC_BRANCH = (
    RHH_PHYSICAL_CLASSIFIED
)

HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED = False
HYPERSURFACE_ALGEBRA_CLOSED = False

DISPERSION_READY = False

assert HYPERSURFACE_ALGEBRA_CLOSED_ON_GENERIC_BRANCH
assert not HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED
assert not HYPERSURFACE_ALGEBRA_CLOSED
assert not DISPERSION_READY

print(
    "HYPERSURFACE_ALGEBRA_CLOSED_ON_GENERIC_BRANCH =",
    HYPERSURFACE_ALGEBRA_CLOSED_ON_GENERIC_BRANCH
)

print(
    "HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED =",
    HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED
)

print(
    "DISPERSION_READY =",
    DISPERSION_READY
)


HYPERSURFACE_ALGEBRA_CLOSED_ON_GENERIC_BRANCH = True
HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED = False
DISPERSION_READY = False



# 15 — Gates de `.1.6.2.1`


In [16]:

GATES = {
    "upstream_full_HH_canonical_bracket_computed":
        UPSTREAM_FULL_HH_CANONICAL_BRACKET_COMPUTED,

    "upstream_RHH_canonical_strong_zero":
        UPSTREAM_RHH_CANONICAL_CLASSIFICATION == "STRONG_ZERO",

    "QJU_reconstructed_from_same_full_field_Hamiltonian":
        QJU_RECONSTRUCTED_FROM_SAME_FULL_FIELD_HAMILTONIAN,

    "full_field_norm_canonical_gradient_explicit":
        FULL_FIELD_NORM_CANONICAL_GRADIENT_EXPLICIT,

    "full_field_primary_to_chi":
        FULL_FIELD_PRIMARY_TO_CHI,

    "full_field_chi_to_psi_explicit":
        FULL_FIELD_CHI_TO_PSI_EXPLICIT,

    "lapse_gradient_cancels_in_chi_stabilization":
        LAPSE_GRADIENT_CANCELS_IN_CHI_STABILIZATION,

    "local_psi_reduction_exact":
        LOCAL_PSI_REDUCTION_EXACT,

    "Delta_FF_compact_operator_form":
        DELTA_FF_COMPACT_OPERATOR_FORM,

    "Delta_local_reduction_exact":
        DELTA_LOCAL_REDUCTION_EXACT,

    "Delta_FF_generic_branch_nonempty":
        DELTA_FF_GENERIC_BRANCH_NONEMPTY,

    "Jacobi_fixes_psi_stabilization_gradient":
        JACOBI_FIXES_PSI_STABILIZATION_GRADIENT,

    "rho_FF_defined_by_exact_stabilization":
        RHO_FF_DEFINED_BY_EXACT_STABILIZATION,

    "full_field_auxiliary_chain_derived_from_complete_QJU":
        FULL_FIELD_AUXILIARY_CHAIN_DERIVED_FROM_COMPLETE_QJU,

    "distributional_kernel_block_structure_explicit":
        DISTRIBUTIONAL_KERNEL_BLOCK_STRUCTURE_EXPLICIT,

    "Delta_FF_multiplicative_block_invertible_on_generic_branch":
        DELTA_FF_MULTIPLICATIVE_BLOCK_INVERTIBLE_ON_GENERIC_BRANCH,

    "distributional_Dirac_block_inverse_regression_pass":
        DISTRIBUTIONAL_DIRAC_BLOCK_INVERSE_REGRESSION_PASS,

    "full_field_distributional_Dirac_kernel_justified":
        FULL_FIELD_DISTRIBUTIONAL_DIRAC_KERNEL_JUSTIFIED,

    "Dirac_HH_correction_weak_zero":
        DIRAC_HH_CORRECTION_WEAK_ZERO,

    "full_HH_Dirac_bracket_computed":
        FULL_HH_DIRAC_BRACKET_COMPUTED,

    "RHH_physical_classified":
        RHH_PHYSICAL_CLASSIFIED,

    "hypersurface_algebra_closed_on_generic_branch":
        HYPERSURFACE_ALGEBRA_CLOSED_ON_GENERIC_BRANCH,

    "hypersurface_algebra_globally_closed":
        HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED,

    "dispersion_ready":
        DISPERSION_READY,
}

for k,vv in GATES.items():
    print(k,":",vv)


upstream_full_HH_canonical_bracket_computed : True
upstream_RHH_canonical_strong_zero : True
QJU_reconstructed_from_same_full_field_Hamiltonian : True
full_field_norm_canonical_gradient_explicit : True
full_field_primary_to_chi : True
full_field_chi_to_psi_explicit : True
lapse_gradient_cancels_in_chi_stabilization : True
local_psi_reduction_exact : True
Delta_FF_compact_operator_form : True
Delta_local_reduction_exact : True
Delta_FF_generic_branch_nonempty : True
Jacobi_fixes_psi_stabilization_gradient : True
rho_FF_defined_by_exact_stabilization : True
full_field_auxiliary_chain_derived_from_complete_QJU : True
distributional_kernel_block_structure_explicit : True
Delta_FF_multiplicative_block_invertible_on_generic_branch : True
distributional_Dirac_block_inverse_regression_pass : True
full_field_distributional_Dirac_kernel_justified : True
Dirac_HH_correction_weak_zero : True
full_HH_Dirac_bracket_computed : True
RHH_physical_classified : True
hypersurface_algebra_closed_on_generic


# 16 — Verdict autorisé

Si tous les asserts passent, le verdict de cette étape est :

\[
\boxed{
\texttt{
PASS-FULL-FIELD-AUXILIARY-CHAIN-
DISTRIBUTIONAL-DIRAC-HH-CORRECTION-
RHH-DIRAC-WEAK-ZERO-GENERIC-BRANCH
}
}
\]

avec :

\[
\boxed{
R_{HH}^{\rm can}=0
}
\]

et :

\[
\boxed{
R_{HH}^{D}\approx0.
}
\]

Le statut physique HH devient alors :

\[
\boxed{
\texttt{
RHH-PHYSICAL-CLASSIFIED-
WEAK-ZERO-GENERIC-FULL-FIELD-DIRAC-BRANCH
}
}
\]

mais les surfaces de dégénérescence restent ouvertes.

Aucune relation de dispersion n'est encore autorisée automatiquement.


In [17]:

if (
    RHH_PHYSICAL_CLASSIFIED
    and
    RHH_DIRAC_CLASSIFICATION
    ==
    "WEAK_ZERO_GENERIC_FULL_FIELD_DIRAC_BRANCH"
):
    FINAL_STATUS = (
        "PASS-FULL-FIELD-AUXILIARY-CHAIN_"
        "DISTRIBUTIONAL-DIRAC-HH-CORRECTION_"
        "RHH-DIRAC-WEAK-ZERO-GENERIC-BRANCH"
    )

    R_HH_OPERATIONAL_STATUS = (
        "WEAK-ZERO-GENERIC-FULL-FIELD-DIRAC-BRANCH"
    )

    NEXT_STATUS = (
        "HH-DIRAC-GENERIC-BRANCH-CLOSED_"
        "NO-DOWNSTREAM-PHYSICS-AUTO-OPENED"
    )
else:
    FINAL_STATUS = (
        "BLOCKED-FULL-FIELD-DIRAC-HH-CLOSURE"
    )

    R_HH_OPERATIONAL_STATUS = (
        "CANONICAL-STRONG-ZERO-DIRAC-PENDING"
    )

    NEXT_STATUS = (
        "NO-DOWNSTREAM-PHYSICS-AUTHORIZED"
    )

print("FINAL_STATUS =",FINAL_STATUS)
print(
    "R_HH_OPERATIONAL_STATUS =",
    R_HH_OPERATIONAL_STATUS
)
print("NEXT_STATUS =",NEXT_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


FINAL_STATUS = PASS-FULL-FIELD-AUXILIARY-CHAIN_DISTRIBUTIONAL-DIRAC-HH-CORRECTION_RHH-DIRAC-WEAK-ZERO-GENERIC-BRANCH
R_HH_OPERATIONAL_STATUS = WEAK-ZERO-GENERIC-FULL-FIELD-DIRAC-BRANCH
NEXT_STATUS = HH-DIRAC-GENERIC-BRANCH-CLOSED_NO-DOWNSTREAM-PHYSICS-AUTO-OPENED
DISPERSION_READY = False



# 17 — Export machine-readable


In [18]:

artifact = {
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.6.2.1",

    "traceability":
        (
            "ACTUAL_QJU_FULL_FIELD_AUXILIARY_CHAIN_"
            "DISTRIBUTIONAL_DIRAC_NO_NEW_PHYSICS"
        ),

    "upstream_1_6_2":
        {
            "executed_sha256":
                UPSTREAM_162_EXECUTED_SHA256,

            "full_HH_canonical_bracket_computed":
                True,

            "RHH_canonical_classification":
                "STRONG_ZERO",

            "R_aux_from_canonical_HH":
                "0",
        },

    "full_field_auxiliary_chain":
        {
            "basis":
                [
                    "p_lambda",
                    "chi",
                    "psi_FF",
                    "rho_FF",
                ],

            "chi":
                "-s^2 + h^{ij} v_i v_j + 1",

            "psi_FF":
                str(psi_FF),

            "Delta_FF":
                "-r^T Q^{-1} r",

            "Delta_FF_witness":
                str(Delta_FF_witness),

            "rho_FF":
                (
                    "A_FF + lambda_mult*Delta_FF, "
                    "with A_FF the N-local coefficient of "
                    "{psi_FF,H0[N]}"
                ),

            "psi_stabilization":
                (
                    "{psi_FF,H[N]} = "
                    "N*rho_FF + D^i(chi) D_i N"
                ),

            "generic_branch":
                "det(Q) != 0 and Delta_FF(x) != 0",
        },

    "distributional_Dirac_kernel":
        {
            "constructed":
                FULL_FIELD_DISTRIBUTIONAL_DIRAC_KERNEL_JUSTIFIED,

            "lower_right_2x2_inverse_block":
                "0",

            "Cinv_33":
                "0",
        },

    "Dirac_HH":
        {
            "computed":
                FULL_HH_DIRAC_BRACKET_COMPUTED,

            "correction":
                "weak_zero",

            "classification":
                RHH_DIRAC_CLASSIFICATION,
        },

    "physical_status":
        {
            "RHH_physical_classified":
                RHH_PHYSICAL_CLASSIFIED,

            "hypersurface_algebra_closed_on_generic_branch":
                HYPERSURFACE_ALGEBRA_CLOSED_ON_GENERIC_BRANCH,

            "hypersurface_algebra_globally_closed":
                HYPERSURFACE_ALGEBRA_GLOBALLY_CLOSED,

            "dispersion_ready":
                False,
        },

    "gates":
        GATES,

    "final_status":
        FINAL_STATUS,

    "R_HH_operational_status":
        R_HH_OPERATIONAL_STATUS,

    "next_status":
        NEXT_STATUS,
}

export_dir = (
    Path("/content/gvh_exports")
    if Path("/content").exists()
    else Path.cwd()/"gvh_exports"
)

export_dir.mkdir(
    parents=True,
    exist_ok=True
)

artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.1.6.2.1_"
    "full_field_Dirac_HH_closure.json"
)

artifact_path.write_text(
    json.dumps(
        artifact,
        indent=2
    ),
    encoding="utf-8"
)

print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.1.6.2.1_full_field_Dirac_HH_closure.json



# Conclusion

Cette étape ne modifie pas le noyau physique.

Elle teste une question unique :

\[
\boxed{
\text{la fermeture canonique forte de `.1.6.2`
survit-elle au passage au crochet de Dirac full-field ?}
}
\]

La chaîne full-field est dérivée comme :

\[
p_\lambda
\to
\chi
\to
\psi_{\rm FF}
\to
\rho_{\rm FF},
\]

avec :

\[
\psi_{\rm FF}
=
-r^TX,
\]

\[
\Delta_{\rm FF}
=
-r^TQ^{-1}r.
\]

La structure distributionnelle du noyau de seconde classe possède un
inverse dès que :

\[
\Delta_{\rm FF}(x)\neq0,
\]

même lorsque les blocs \(\mathsf X,\mathsf Y\) sont des opérateurs
différentiels non commutatifs.

Le bloc inférieur droit de \(C^{-1}\) est nul.

Sur la surface seconde-classe :

\[
a_A[N]
\approx
(0,0,0,\zeta_N),
\]

donc la correction HH de Dirac est faiblement nulle.

Si l'exécution passe :

\[
\boxed{
R_{HH}^{\rm can}=0,
\qquad
R_{HH}^{D}\approx0
}
\]

sur la branche générique :

\[
\det Q\neq0,
\qquad
\Delta_{\rm FF}\neq0.
\]

Les branches dégénérées restent explicitement non classifiées et :

\[
\boxed{
\mathrm{DISPERSION\_READY=False}.
}
\]
